# Deploying Mistral Small 4 119B with TensorRT-LLM

This notebook walks you through serving `mistralai/Mistral-Small-4-119B-2603` with TensorRT-LLM using the AutoDeploy path added in this branch.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference on NVIDIA GPUs. In this branch, Mistral Small 4 support is enabled through AutoDeploy with a temporary local tokenizer and processor bridge.

**Model Resources:**
- [Hugging Face model card](https://huggingface.co/mistralai/Mistral-Small-4-119B-2603)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)
- [AutoDeploy guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html)

**Branch-Specific Notes:**
- Use `examples/auto_deploy/model_registry/configs/mistral_small_4_119b.yaml` for standalone `trtllm-serve`.
- That YAML sets `world_size: 8` in the standalone variants; adjust the config if your GPU topology differs.
- The branch also includes `mistral_small_4_119b_no_moe_fusion.yaml` and `mistral_small_4_119b_lite.yaml` for debugging and reduced-layer sanity runs.


## Prerequisites & Environment

Set up a containerized TensorRT-LLM environment by running the following command in a terminal:

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all -p 8000:8000 nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc1
```

Inside the container, start from the branch checkout that contains the Mistral Small 4 AutoDeploy onboarding files.


In [ ]:
# If pip is not available
!python -m ensurepip --default-pip

In [ ]:
%pip install torch openai pillow

## Verify GPU

Check that CUDA is available and the GPUs are visible before launching the server.


In [1]:
import sys

import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

Python: 3.12.3 (main, Nov  6 2025, 13:44:16) [GCC 13.3.0]
CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA H100 80GB HBM3
GPU[1]: NVIDIA H100 80GB HBM3
GPU[2]: NVIDIA H100 80GB HBM3
GPU[3]: NVIDIA H100 80GB HBM3
GPU[4]: NVIDIA H100 80GB HBM3
GPU[5]: NVIDIA H100 80GB HBM3
GPU[6]: NVIDIA H100 80GB HBM3
GPU[7]: NVIDIA H100 80GB HBM3


## OpenAI-Compatible Server

Start a local OpenAI-compatible server with TensorRT-LLM from the branch checkout.

The Mistral Small 4 onboarding in this branch relies on:
- `examples/auto_deploy/model_registry/configs/mistral_small_4_119b.yaml`

Run the following command from the docker terminal:


### Load the Model

```shell
trtllm-serve "mistralai/Mistral-Small-4-119B-2603" \
  --host 0.0.0.0 \
  --port 8000 \
  --backend _autodeploy \
  --trust_remote_code \
  --extra_llm_api_options examples/auto_deploy/model_registry/configs/mistral_small_4_119b.yaml
```


Your server is now running.


## Use the API

Use the OpenAI-compatible client to send requests to the TensorRT-LLM server. Although Mistral Small 4 supports multimodal input, this cookbook focuses on text-only prompts.


In [1]:
from openai import OpenAI

BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
MODEL_ID = "mistralai/Mistral-Small-4-119B-2603"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

In [2]:
# Basic chat completion
print("Chat Completion Example")
print("=" * 50)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 15% of 85? Show your reasoning."},
    ],
    temperature=1,
    top_p=0.95,
    max_tokens=512,
)

print("Response:")
print(response.choices[0].message.content)

Chat Completion Example
Response:
To find 15% of 85, follow these steps:

1. **Understand what percentage means:**
   - "Percent" means per hundred, so 15% is the same as 15/100 or 0.15.

2. **Convert the percentage to a decimal:**
   - 15% = 0.15

3. **Multiply the decimal by the number you want to find the percentage of:**
   - 0.15 × 85

4. **Perform the multiplication:**
   - 0.15 × 85 = 12.75

So, 15% of 85 is **12.75**.


In [3]:
# Streaming chat completion
print("Streaming response:")
print("=" * 50)

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"},
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Streaming response:


The first 5 prime numbers are:

2, 3, 5, 7, 11

## Additional Resources

- [Mistral Small 4 119B on Hugging Face](https://huggingface.co/mistralai/Mistral-Small-4-119B-2603)
- [AutoDeploy guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html)
- [`examples/auto_deploy/model_registry/configs/mistral_small_4_119b.yaml`](../model_registry/configs/mistral_small_4_119b.yaml)
